# Final models

Fit the final forecasting models and compare each one with persistence and AR(1) on the training and test sets.

In [13]:
import pandas as pd
import numpy as np

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import ElasticNet, HuberRegressor, Lasso, LinearRegression, Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler

## Data preparation

In [14]:
train_data = (
    pd.read_parquet("../data/train.parquet")
    .sort_values("date")
    .reset_index(drop=True)
)
test_data = (
    pd.read_parquet("../data/test.parquet")
    .sort_values("date")
    .reset_index(drop=True)
)

for data in (train_data, test_data):
    data["usd_zar_28_movement"] = data["usd_zar_28"] - data["usd_zar"]

# Engineer train and test together so the first test rows can use training history
# for lagged and rolling features.
all_data = pd.concat(
    [train_data.assign(_split="train"), test_data.assign(_split="test")],
    ignore_index=True,
).sort_values("date").reset_index(drop=True)

all_data["interest_rate_diff"] = all_data["sa_repo_rate"] - all_data["us_fed_funds"]
all_data["policy_rate_differential"] = all_data["interest_rate_diff"]
all_data["sa_us_5y_yield_spread"] = all_data["sa_5y_yield"] - all_data["us_5y_yield"]

log_usd_zar = np.log(all_data["usd_zar"])
all_data["fx_log_return_5d"] = log_usd_zar.diff(5)
all_data["fx_log_return_21d"] = log_usd_zar.diff(21)
all_data["fx_realized_vol_21d"] = log_usd_zar.diff().shift(1).rolling(21).std()
all_data["fx_realized_vol_63d"] = log_usd_zar.diff().shift(1).rolling(63).std()
all_data["sa_cpi_log_change_21d"] = np.log(all_data["sa_cpi"]).diff(21)
all_data["sa_real_gdp_log_change_63d"] = np.log(all_data["sa_real_gdp"]).diff(63)
all_data["commodities"] = all_data[
    [
        "iron_ore_usd_per_tonne",
        "gold_usd_per_oz",
        "platinum_usd_per_oz",
        "richards_bay_coal_usd",
    ]
].mean(axis=1)

engineered_train = all_data.loc[all_data["_split"].eq("train")].drop(columns="_split").reset_index(drop=True)
engineered_test = all_data.loc[all_data["_split"].eq("test")].drop(columns="_split").reset_index(drop=True)

y_train = engineered_train["usd_zar_28_movement"]
y_test = engineered_test["usd_zar_28_movement"]

## AR(1) benchmark

In [15]:
AR1_HORIZON = 28

# Fit log(USD/ZAR)_t = intercept + phi * log(USD/ZAR)_(t-1)
# using training observations only.
train_log_usd_zar = np.log(engineered_train["usd_zar"])
ar1_model = LinearRegression()
ar1_model.fit(
    train_log_usd_zar.shift(1).iloc[1:].to_frame("lagged_log_usd_zar"),
    train_log_usd_zar.iloc[1:],
)


def ar1_movement_forecast(spot, horizon=AR1_HORIZON):
    """Return the AR(1) forecast as movement from the current spot level."""
    phi = float(ar1_model.coef_[0])
    intercept = float(ar1_model.intercept_)
    log_spot = np.log(np.asarray(spot))
    if np.isclose(phi, 1.0):
        log_forecast = log_spot + intercept * horizon
    else:
        log_forecast = phi**horizon * log_spot + intercept * (1 - phi**horizon) / (1 - phi)
    return np.exp(log_forecast) - np.asarray(spot)


def evaluate_model(model_name, model, features, data, target):
    """Compare movement RMSE for a fitted model, persistence, and AR(1)."""
    valid_rows = data[features].notna().all(axis=1) & target.notna()
    actual = target.loc[valid_rows]
    model_prediction = model.predict(data.loc[valid_rows, features])
    persistence_prediction = np.zeros(len(actual))
    ar1_prediction = ar1_movement_forecast(data.loc[valid_rows, "usd_zar"])

    return pd.DataFrame(
        {
            "RMSE": [
                root_mean_squared_error(actual, model_prediction),
                root_mean_squared_error(actual, persistence_prediction),
                root_mean_squared_error(actual, ar1_prediction),
            ]
        },
        index=[model_name, "Persistence", "AR(1)"],
    ).sort_values("RMSE")

## LightGBM

In [16]:
lgbm_features = [
    "gold_usd_per_oz",
    "usd_zar_1w_return",
    "usd_zar_1m_volatility",
    "interest_rate_diff",
]

lgbm_model = LGBMRegressor(
    n_estimators=150,
    learning_rate=0.05,
    num_leaves=15,
    random_state=42,
    verbosity=-1,
    n_jobs=1,
)
lgbm_model.fit(engineered_train[lgbm_features], y_train)

,num_leaves,15
,learning_rate,0.05
,n_estimators,150
,random_state,42
,n_jobs,1
,verbosity,-1
,boosting_type,'gbdt'
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None


In [17]:
# Training data: LightGBM vs persistence and AR(1)
evaluate_model("LightGBM", lgbm_model, lgbm_features, engineered_train, y_train)

,RMSE
LightGBM,0.306231
AR(1),0.521657
Persistence,0.524441


In [18]:
# Test data: LightGBM vs persistence and AR(1)
evaluate_model("LightGBM", lgbm_model, lgbm_features, engineered_test, y_test)

,RMSE
AR(1),0.544117
Persistence,0.544335
LightGBM,0.672096


## XGBoost

In [19]:
xgb_features = lgbm_features

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=5,
    gamma=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=-1,
)
xgb_model.fit(engineered_train[xgb_features], y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [20]:
# Training data: XGBoost vs persistence and AR(1)
evaluate_model("XGBoost", xgb_model, xgb_features, engineered_train, y_train)

,RMSE
XGBoost,0.338730
AR(1),0.521657
Persistence,0.524441


In [21]:
# Test data: XGBoost vs persistence and AR(1)
evaluate_model("XGBoost", xgb_model, xgb_features, engineered_test, y_test)

,RMSE
AR(1),0.544117
Persistence,0.544335
XGBoost,0.661908


## Lasso

In [22]:
lasso_features = [
    "usd_zar",
    "brent_usd_per_barrel",
    "interest_rate_diff",
    "sa_us_5y_yield_spread",
    "sa_yoy_inflation",
    "sa_5y_cds_bp",
    "vix",
    "broad_usd_index",
    "sa_cpi",
    "usd_zar_1w_return",
    "usd_zar_1m_return",
    "usd_zar_3m_return",
    "usd_zar_1m_volatility",
]

lasso_model = make_pipeline(
    StandardScaler(),
    Lasso(alpha=0.1, max_iter=50_000),
)
lasso_model.fit(engineered_train[lasso_features], y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('lasso', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['usd_zar','brent_usd_per_barrel','interest_rate_diff',..., 'usd_zar_1m_return','usd_zar_3m_return','usd_zar_1m_volatility']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [23]:
# Training data: Lasso vs persistence and AR(1)
evaluate_model("Lasso", lasso_model, lasso_features, engineered_train, y_train)

,RMSE
AR(1),0.521657
Lasso,0.522753
Persistence,0.524441


In [24]:
# Test data: Lasso vs persistence and AR(1)
evaluate_model("Lasso", lasso_model, lasso_features, engineered_test, y_test)

,RMSE
Lasso,0.544020
AR(1),0.544117
Persistence,0.544335


## Multi-Layer Perceptron

In [25]:
mlp_features = [
    "gold_usd_per_oz_return",
    "sa_yoy_inflation",
    "usd_zar_1m_return",
    "usd_zar_1m_volatility",
]

mlp_model = make_pipeline(
    StandardScaler(),
    MLPRegressor(
        hidden_layer_sizes=(128, 64, 32),
        alpha=0.01,
        learning_rate="adaptive",
        max_iter=1000,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=30,
        random_state=42,
    ),
)
mlp_model.fit(engineered_train[mlp_features], y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('mlpregressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['gold_usd_per_oz_return','sa_yoy_inflation','usd_zar_1m_return', 'usd_zar_1m_volatility']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [26]:
# Training data: MLP vs persistence and AR(1)
evaluate_model("MLP", mlp_model, mlp_features, engineered_train, y_train)

,RMSE
MLP,0.378550
AR(1),0.521657
Persistence,0.524441


In [27]:
# Test data: MLP vs persistence and AR(1)
evaluate_model("MLP", mlp_model, mlp_features, engineered_test, y_test)

,RMSE
AR(1),0.544117
Persistence,0.544335
MLP,0.652374


## OLS

In [28]:
ols_features = [
    "broad_usd_index",
    "brent_usd_per_barrel",
    "sa_cpi",
    "usd_zar_1m_return",
    "fx_log_return_5d",
    "fx_log_return_21d",
    "sa_cpi_log_change_21d",
    "sa_real_gdp_log_change_63d",
]

ols_model = make_pipeline(StandardScaler(), LinearRegression())
ols_rows = engineered_train[ols_features].notna().all(axis=1)
ols_model.fit(engineered_train.loc[ols_rows, ols_features], y_train.loc[ols_rows])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('linearregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['broad_usd_index','brent_usd_per_barrel','sa_cpi',...,'fx_log_return_21d', 'sa_cpi_log_change_21d','sa_real_gdp_log_change_63d']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [29]:
# Training data: OLS vs persistence and AR(1)
evaluate_model("OLS", ols_model, ols_features, engineered_train, y_train)

,RMSE
OLS,0.511429
AR(1),0.520429
Persistence,0.523316


In [30]:
# Test data: OLS vs persistence and AR(1)
evaluate_model("OLS", ols_model, ols_features, engineered_test, y_test)

,RMSE
AR(1),0.544117
Persistence,0.544335
OLS,0.614477


## Ridge

In [31]:
ridge_features = [
    "sa_repo_rate",
    "sa_real_gdp",
    "policy_rate_differential",
    "sa_us_5y_yield_spread",
    "sa_cpi_log_change_21d",
]

ridge_model = make_pipeline(StandardScaler(), Ridge(alpha=10.0))
ridge_rows = engineered_train[ridge_features].notna().all(axis=1)
ridge_model.fit(engineered_train.loc[ridge_rows, ridge_features], y_train.loc[ridge_rows])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('ridge', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['sa_repo_rate','sa_real_gdp','policy_rate_differential', 'sa_us_5y_yield_spread','sa_cpi_log_change_21d']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [32]:
# Training data: Ridge vs persistence and AR(1)
evaluate_model("Ridge", ridge_model, ridge_features, engineered_train, y_train)

,RMSE
Ridge,0.506723
AR(1),0.521128
Persistence,0.523839


In [33]:
# Test data: Ridge vs persistence and AR(1)
evaluate_model("Ridge", ridge_model, ridge_features, engineered_test, y_test)

,RMSE
AR(1),0.544117
Persistence,0.544335
Ridge,0.548637


## Elastic Net

In [34]:
elastic_net_features = [
    "sa_real_gdp",
    "sa_5y_cds_bp",
    "sa_5y_yield",
    "policy_rate_differential",
    "fx_log_return_5d",
    "fx_realized_vol_21d",
]

elastic_net_model = TransformedTargetRegressor(
    regressor=make_pipeline(
        StandardScaler(),
        ElasticNet(alpha=0.02, l1_ratio=0.35, max_iter=20_000, random_state=42),
    ),
    transformer=StandardScaler(),
)
elastic_net_rows = engineered_train[elastic_net_features].notna().all(axis=1)
elastic_net_model.fit(
    engineered_train.loc[elastic_net_rows, elastic_net_features],
    y_train.loc[elastic_net_rows],
)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",Pipeline(step...m_state=42))])
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",StandardScaler()
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",None
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](6,)","['sa_real_gdp','sa_5y_cds_bp','sa_5y_yield','policy_rate_differential', 'fx_log_return_5d','fx_realized_vol_21d']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,6
regressor_ regressor_: objectFitted regressor.,Pipeline,Pipeline(step...m_state=42))])
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,StandardScaler,StandardScaler()
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('elasticnet', ...)]"


In [35]:
# Training data: Elastic Net vs persistence and AR(1)
evaluate_model(
    "Elastic Net",
    elastic_net_model,
    elastic_net_features,
    engineered_train,
    y_train,
)

,RMSE
Elastic Net,0.496397
AR(1),0.521203
Persistence,0.523910


In [36]:
# Test data: Elastic Net vs persistence and AR(1)
evaluate_model(
    "Elastic Net",
    elastic_net_model,
    elastic_net_features,
    engineered_test,
    y_test,
)

,RMSE
AR(1),0.544117
Persistence,0.544335
Elastic Net,0.554314


## Huber

In [37]:
huber_features = [
    "sa_repo_rate",
    "usd_zar_1m_return",
    "sa_cpi_log_change_21d",
]

huber_model = TransformedTargetRegressor(
    regressor=make_pipeline(
        StandardScaler(),
        HuberRegressor(epsilon=1.35, alpha=0.01, max_iter=2_000),
    ),
    transformer=StandardScaler(),
)
huber_rows = engineered_train[huber_features].notna().all(axis=1)
huber_model.fit(
    engineered_train.loc[huber_rows, huber_features],
    y_train.loc[huber_rows],
)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",Pipeline(step..._iter=2000))])
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",StandardScaler()
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",None
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](3,)","['sa_repo_rate','usd_zar_1m_return','sa_cpi_log_change_21d']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,3
regressor_ regressor_: objectFitted regressor.,Pipeline,Pipeline(step..._iter=2000))])
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,StandardScaler,StandardScaler()
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('huberregressor', ...)]"


In [38]:
# Training data: Huber vs persistence and AR(1)
evaluate_model("Huber", huber_model, huber_features, engineered_train, y_train)

,RMSE
AR(1),0.521128
Huber,0.523688
Persistence,0.523839


In [39]:
# Test data: Huber vs persistence and AR(1)
evaluate_model("Huber", huber_model, huber_features, engineered_test, y_test)

,RMSE
Huber,0.540929
AR(1),0.544117
Persistence,0.544335


## Spline Ridge

In [40]:
spline_ridge_features = [
    "iron_ore_usd_per_tonne",
    "sa_repo_rate",
    "sa_cpi",
    "sa_yoy_inflation",
    "usd_zar_1w_return",
    "usd_zar_1m_return",
    "sa_5y_yield",
    "interest_rate_diff",
    "fx_realized_vol_63d",
]

spline_ridge_model = make_pipeline(
    StandardScaler(),
    SplineTransformer(n_knots=4, degree=2, include_bias=False),
    Ridge(alpha=25.0),
)
spline_ridge_rows = engineered_train[spline_ridge_features].notna().all(axis=1)
spline_ridge_model.fit(
    engineered_train.loc[spline_ridge_rows, spline_ridge_features],
    y_train.loc[spline_ridge_rows],
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('splinetransformer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['iron_ore_usd_per_tonne','sa_repo_rate','sa_cpi',...,'sa_5y_yield', 'interest_rate_diff','fx_realized_vol_63d']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [41]:
# Training data: Spline Ridge vs persistence and AR(1)
evaluate_model(
    "Spline Ridge",
    spline_ridge_model,
    spline_ridge_features,
    engineered_train,
    y_train,
)

,RMSE
Spline Ridge,0.477664
AR(1),0.520351
Persistence,0.523211


In [42]:
# Test data: Spline Ridge vs persistence and AR(1)
evaluate_model(
    "Spline Ridge",
    spline_ridge_model,
    spline_ridge_features,
    engineered_test,
    y_test,
)

,RMSE
AR(1),0.544117
Persistence,0.544335
Spline Ridge,0.562254
